In [10]:
import os
import re
import getpass
import requests
import openai
from openai import OpenAI

In [48]:
import os
import json
from typing import List, Optional, Dict, Any

import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pydantic import BaseModel, Field

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ==== НАСТРОЙКИ ====
OPENAI_API_KEY = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://api.vsegpt.ru/v1"
OPENAI_MODEL = "openai/gpt-4o-mini"

# Пути к файлам
CSV_CRITERIA_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv"  # файл во вложении
DOCX_SPEC_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\ТЗ Электронный журнал_v2.1.docx"
OUTPUT_REPORT_DOCX = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Analysis_Report_SGR.docx"

if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_OPENAI_API_KEY_HERE":
    raise ValueError("Не задан OPENAI_API_KEY. Установите переменную окружения или впишите ключ.")

print("✅ Конфигурация загружена")


✅ Конфигурация загружена


In [27]:
from pydantic import BaseModel, Field
class CriterionReasoning(BaseModel):
    """Пошаговое рассуждение для одного критерия (Schema-Guided Reasoning)."""
    
    criterion_id: int = Field(..., description="Порядковый номер критерия")
    criterion_name: str = Field(..., description="Название критерия")
    criterion_description: str = Field(..., description="Описание из CSV")
    importance_level: int = Field(..., description="Уровень важности (2 или 3)")

    criterion_understanding: str = Field(
        ...,
        description="Объясни, ЧТО проверяет этот критерий (1–2 предложения)."
    )

    relevant_sections: str = Field(
        ...,
        description="Какие разделы/приложения ТЗ релевантны для этого критерия?"
    )

    reasoning_steps: str = Field(
        ...,
        description="Пошагово объясни: Критерий требует X → В документе найдено Y → Вывод Z."
    )

    status: str = Field(
        ...,
        description="Статус выполнения критерия.",
        enum=["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]
    )

    quote_or_evidence: str = Field(
        ...,
        description="Прямая цитата из ТЗ или обоснование статуса."
    )

    # --- ИСПРАВЛЕНИЯ НИЖЕ (убраны default=None) ---
    
    recommendation: str = Field(
        ..., # Обязательно (Three dots) означает, что поле должно быть в JSON (даже если null)
        description="Рекомендация по улучшению. Если нет — передать null."
    )

    confidence_score: float = Field(
        ..., 
        ge=0.0,
        le=1.0,
        description="Уверенность модели в выводе (0-1). Если не применимо — передать null."
    )

    confidence_explanation: str = Field(
        ...,
        description="Почему такая уверенность? Если не применимо — передать null."
    )


class SpecificationAnalysisWithReasoning(BaseModel):
    """Анализ ТЗ с явным SGR для каждого критерия."""
    
    reasoning_schema_used: bool = Field(
        ..., # Убрали default=True, модель должна сама решить или вы жестко задаете это в промпте
        description="Флаг: используется ли Schema-Guided Reasoning."
    )

    overall_summary: str = Field(
        ...,
        description="Общая оценка качества ТЗ (1-2 абзаца)."
    )

    criteria_analysis: List[CriterionReasoning] = Field(
        ...,
        description="Анализ каждого критерия с пошаговым рассуждением."
    )

    # Лучше временно убрать Dict[str, Any] или заменить на конкретную модель, 
    # так как 'Any' плохо работает со строгим режимом.
    # Если метрики не критичны, можно закомментировать поле metrics:
    # metrics: Dict[str, str] = Field(..., description="Метрики анализа (ключ-значение).")


C:\Users\troyd\AppData\Local\Temp\ipykernel_12388\2543937392.py:25: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  status: str = Field(


In [28]:
def load_criteria_from_csv(csv_path: str) -> List[Dict[str, Any]]:
    """
    Читает критерии из CSV с разделителем ';' (точка с запятой).
    Ожидаемые колонки: Название, Описание, Важность
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV с критериями не найден: {csv_path}")
    
    # Важно: использовать sep=';' и encoding зависит от системы
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    
    print(f"Загруженные колонки: {df.columns.tolist()}")
    print(f"Первая строка: {df.iloc[0].to_dict()}")
    
    # Проверка обязательных колонок
    required_cols = ["Название", "Описание", "Важность"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"В CSV отсутствует обязательная колонка '{col}'. Колонки: {df.columns.tolist()}")
    
    criteria = []
    for idx, row in df.iterrows():
        criteria.append({
            "id": idx + 1,  # Порядковый номер
            "name": str(row["Название"]).strip(),
            "description": str(row["Описание"]).strip(),
            "importance": int(row["Важность"]),
        })
    
    print(f"✅ Загружено критериев: {len(criteria)}")
    return criteria


def extract_text_from_docx(docx_path: str) -> str:
    """
    Извлекает текст из DOCX файла.
    """
    if not os.path.exists(docx_path):
        raise FileNotFoundError(f"DOCX файл не найден: {docx_path}")
    
    doc = Document(docx_path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    full_text = "\n".join(paragraphs)
    
    print(f"✅ Текст ТЗ извлечён ({len(full_text)} символов)")
    return full_text


In [29]:
def create_sgr_prompt(spec_text: str, criteria: List[Dict[str, Any]]) -> str:
    """
    Создаёт prompt с явной схемой рассуждения (SGR).
    """
    criteria_list = "\n".join([
        f"{c['id']}. [{c['importance']}] {c['name']}\n   {c['description']}"
        for c in criteria
    ])

    sgr_prompt = f"""
Ты — эксперт по анализу технических заданий (ТЗ) информационных систем.
Твоя задача — проанализировать ТЗ по заданным критериям, строго следуя Schema-Guided Reasoning (SGR).

════════════════════════════════════════
ИНСТРУКЦИЯ ПО СХЕМЕ РАССУЖДЕНИЯ (ОБЯЗАТЕЛЬНО!):
════════════════════════════════════════

Для каждого критерия ты ДОЛЖЕН выполнить шаги 1–6 в строгом порядке:

1️⃣ ПОНИМАНИЕ (criterion_understanding)
   Объясни, ЧТО ИМЕННО проверяет этот критерий на основе его названия и описания.
   Что является признаком "выполнения" этого критерия? (1–2 предложения)

2️⃣ ПОИСК РЕЛЕВАНТНЫХ ЧАСТЕЙ (relevant_sections)
   Какие разделы, приложения, таблицы или структурные элементы ТЗ имеют отношение к этому критерию?
   Перечисли их конкретно.

3️⃣ ЛОГИЧЕСКОЕ РАССУЖДЕНИЕ (reasoning_steps)
   Логическая цепочка:
     "Критерий требует [X] → В документе я нашёл [Y] → Это означает [Z]"
   Будь конкретен, ссылайся на фактические части документа.

4️⃣ ОПРЕДЕЛЕНИЕ СТАТУСА (status) — РОВНО ОДИН из трёх:
   - "✅ Выполнено"      (критерий полностью адресован в ТЗ)
   - "⚠️ Частично"        (критерий частично описан, есть недостатки или неясности)
   - "❌ Не выполнено"    (критерий отсутствует в ТЗ)

5️⃣ ДОКАЗАТЕЛЬСТВО (quote_or_evidence)
   Привести прямую цитату из ТЗ (короче, ~1 строка) или конкретное обоснование статуса.

6️⃣ РЕКОМЕНДАЦИЯ (recommendation)
   Если критерий выполнен не полностью — что нужно улучшить/дописать?
   Если критерий выполнен полностью — напиши "Нет".

7️⃣ УВЕРЕННОСТЬ (confidence_score, confidence_explanation)
   confidence_score: число 0.0–1.0 (0 = сомневаюсь, 1 = уверен)
   confidence_explanation: краткое объяснение, почему именно такая уверенность?

════════════════════════════════════════
КРИТЕРИИ (из CSV):
════════════════════════════════════════
{criteria_list}

════════════════════════════════════════
ТЕХНИЧЕСКОЕ ЗАДАНИЕ (для анализа):
════════════════════════════════════════
{spec_text}

════════════════════════════════════════
ТРЕБОВАНИЕ К ВЫВОДУ:
════════════════════════════════════════
Верни ровно ОДИН JSON-объект, строго соответствующий предоставленной JSON-схеме.
Анализ КАЖДОГО критерия должен точно следовать 7 шагам выше.
Не добавляй никакого текста вне JSON.
"""
    return sgr_prompt


In [47]:
class SpecAnalyzerWithSGR:
    """Анализатор ТЗ с Schema-Guided Reasoning через VseGPT API."""
    
    def __init__(self, api_key: str, model: str = "openai/gpt-4o-mini"):
        """
        Инициализация анализатора с VseGPT.
        
        Args:
            api_key: API ключ от VseGPT
            model: Модель (по умолчанию openai/gpt-4o-mini)
        """
        self.client = OpenAI(
            api_key=OPENAI_API_KEY,
            base_url="https://api.vsegpt.ru/v1"
        )
        self.model = model
        print(f"✅ Инициализирован анализатор VseGPT")
        print(f"   Модель: {self.model}")
        print(f"   Base URL: https://api.vsegpt.ru/v1")

    def analyze_with_sgr(
        self,
        spec_text: str,
        criteria: List[Dict[str, Any]]
    ) -> SpecificationAnalysisWithReasoning:
        """
        Выполняет анализ ТЗ с SGR через VseGPT API.
        """
        prompt = create_sgr_prompt(spec_text, criteria)

        json_schema = {
            "name": "SpecificationAnalysisWithReasoning",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "reasoning_schema_used": {
                        "type": "boolean"
                    },
                    "overall_summary": {
                        "type": "string"
                    },
                    "criteria_analysis": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "criterion_id": {"type": "integer"},
                                "criterion_name": {"type": "string"},
                                "criterion_description": {"type": "string"},
                                "importance_level": {"type": "integer"},
                                "criterion_understanding": {"type": "string"},
                                "relevant_sections": {"type": "string"},
                                "reasoning_steps": {"type": "string"},
                                "status": {
                                    "type": "string",
                                    "enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]
                                },
                                "quote_or_evidence": {"type": "string"},
                                "recommendation": {
                                    "type": ["string", "null"]
                                },
                                "confidence_score": {
                                    "type": ["number", "null"],
                                    "minimum": 0.0,
                                    "maximum": 1.0
                                },
                                "confidence_explanation": {
                                    "type": ["string", "null"]
                                }
                            },
                            "required": [
                                "criterion_id", "criterion_name", "criterion_description",
                                "importance_level", "criterion_understanding",
                                "relevant_sections", "reasoning_steps", "status",
                                "quote_or_evidence", "recommendation", "confidence_score",
                                "confidence_explanation"
                            ],
                            "additionalProperties": False
                        }
                    },
                    "metrics": {
                        "type": "object",
                        "properties": {},
                        "additionalProperties": False  
                    }
        },
        "required": ["reasoning_schema_used", "overall_summary", "criteria_analysis", "metrics"],
        "additionalProperties": False
    }
}

        print(f"\n📤 Отправляю запрос к VseGPT...")
        print(f"   Количество критериев: {len(criteria)}")
        print(f"   Размер ТЗ: {len(spec_text)} символов")
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                max_tokens=8000,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": json_schema
                }
            )

            raw_json = response.choices[0].message.content
            
            print(f"✅ Получен ответ от VseGPT")
            print(f"   Размер ответа: {len(raw_json)} символов")
            
            # Парсим JSON
            data = json.loads(raw_json)
            analysis = SpecificationAnalysisWithReasoning(**data)
            
            print(f"✅ Анализ успешно распарсен")
            print(f"   Проанализировано критериев: {len(analysis.criteria_analysis)}")
            
            return analysis
            
        except json.JSONDecodeError as e:
            print(f"❌ Ошибка парсинга JSON: {e}")
            print(f"\nПервые 500 символов ответа:")
            print(raw_json[:500])
            raise
            
        except Exception as e:
            print(f"❌ Ошибка при вызове VseGPT API")
            print(f"   Тип ошибки: {type(e).__name__}")
            print(f"   Описание: {str(e)}")
            raise


In [41]:
def export_sgr_analysis_to_docx(
    analysis: SpecificationAnalysisWithReasoning,
    source_docx_path: str,
    output_path: str
):
    """
    Экспортирует результат SGR-анализа в DOCX-отчёт.
    """
    doc = Document()

    # Заголовок
    title = doc.add_heading("Отчёт по анализу ТЗ", level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(f"Исходный документ: {os.path.basename(source_docx_path)}")
    doc.add_paragraph(" ")

    # Общая оценка
    doc.add_heading("1. Общая оценка", level=2)
    doc.add_paragraph(analysis.overall_summary)

    # Метрики
    '''if analysis.metrics:
        doc.add_heading("2. Метрики", level=2)
        for k, v in analysis.metrics.items():
            doc.add_paragraph(f"• {k}: {v}")
    doc.add_paragraph(" ")'''

    # Детальный анализ
    doc.add_heading("3. Детальный анализ по критериям", level=2)

    for crit in analysis.criteria_analysis:
        # Заголовок критерия
        heading_text = f"3.{crit.criterion_id} {crit.criterion_name} (Важность: {crit.importance_level})"
        doc.add_heading(heading_text, level=3)

        # Статус (жирный)
        p = doc.add_paragraph()
        run = p.add_run(f"Статус: ")
        run.bold = True
        p.add_run(crit.status)

        # Описание из CSV (серый текст)
        doc.add_paragraph(f"Определение: {crit.criterion_description}", style="List Bullet")

        # Понимание
        p = doc.add_paragraph()
        run = p.add_run("Понимание критерия: ")
        run.bold = True
        doc.add_paragraph(crit.criterion_understanding)

        # Релевантные разделы
        p = doc.add_paragraph()
        run = p.add_run("Релевантные разделы ТЗ: ")
        run.bold = True
        doc.add_paragraph(crit.relevant_sections)

        # Рассуждение
        p = doc.add_paragraph()
        run = p.add_run("Логическое рассуждение: ")
        run.bold = True
        doc.add_paragraph(crit.reasoning_steps)

        # Доказательство
        p = doc.add_paragraph()
        run = p.add_run("Доказательство / цитата: ")
        run.bold = True
        doc.add_paragraph(crit.quote_or_evidence)

        # Рекомендация (если есть)
        if crit.recommendation and crit.recommendation.lower() != "нет":
            p = doc.add_paragraph()
            run = p.add_run("Рекомендация: ")
            run.bold = True
            doc.add_paragraph(crit.recommendation)

        # Уверенность
        if crit.confidence_score is not None:
            doc.add_paragraph(f"Уверенность модели: {crit.confidence_score:.1%}")
        if crit.confidence_explanation:
            doc.add_paragraph(f"Комментарий: {crit.confidence_explanation}")

        doc.add_paragraph(" ")  # Разделитель

    # Сохранение
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    doc.save(output_path)
    print(f"✅ Отчёт сохранён: {output_path}")


In [49]:
print("=" * 60)
print("АНАЛИЗ ТЗ С SCHEMA-GUIDED REASONING")
print("=" * 60)

# 1. Загружаем критерии из CSV
try:
    criteria = load_criteria_from_csv(CSV_CRITERIA_PATH)
    print(f"Первый критерий: {criteria[0]}")
except Exception as e:
    print(f"❌ Ошибка при чтении CSV: {e}")
    raise

# 2. Извлекаем текст ТЗ
try:
    spec_text = extract_text_from_docx(DOCX_SPEC_PATH)
    print(f"Первые 200 символов ТЗ: {spec_text[:200]}...")
except Exception as e:
    print(f"❌ Ошибка при чтении DOCX: {e}")
    raise

# 3. Создаём анализатор и запускаем анализ
try:
    analyzer_sgr = SpecAnalyzerWithSGR(
        api_key=OPENAI_API_KEY,
        model=OPENAI_MODEL
    )
    
    analysis_result = analyzer_sgr.analyze_with_sgr(
        spec_text=spec_text,
        criteria=criteria
    )
except Exception as e:
    print(f"❌ Ошибка при анализе: {e}")
    raise

# 4. Вывод в консоль (первые 3 критерия)
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ АНАЛИЗА (первые 3 критерия)")
print("=" * 60)

for crit in analysis_result.criteria_analysis[:3]:
    print(f"\n📌 Критерий {crit.criterion_id}: {crit.criterion_name}")
    print(f"   Статус: {crit.status}")
    print(f"   Уверенность: {crit.confidence_score:.1%}" if crit.confidence_score else "")
    print(f"   Рассуждение: {crit.reasoning_steps[:100]}...")

# 5. Экспорт в DOCX
try:
    export_sgr_analysis_to_docx(
        analysis=analysis_result,
        source_docx_path=DOCX_SPEC_PATH,
        output_path=OUTPUT_REPORT_DOCX
    )
except Exception as e:
    print(f"❌ Ошибка при экспорте: {e}")
    raise

print("\n" + "=" * 60)
print("✅ ВСЕ ОПЕРАЦИИ ЗАВЕРШЕНЫ УСПЕШНО!")
print("=" * 60)


АНАЛИЗ ТЗ С SCHEMA-GUIDED REASONING
Загруженные колонки: ['Название', 'Описание', 'Важность']
Первая строка: {'Название': 'Полнота функциональных требований', 'Описание': 'Оценка того, насколько подробно описано, что должна делать система, включая объекты, бизнес-логику, роли и отчётность.', 'Важность': 3}
✅ Загружено критериев: 26
Первый критерий: {'id': 1, 'name': 'Полнота функциональных требований', 'description': 'Оценка того, насколько подробно описано, что должна делать система, включая объекты, бизнес-логику, роли и отчётность.', 'importance': 3}
✅ Текст ТЗ извлечён (122260 символов)
Первые 200 символов ТЗ: У Т В Е Р Ж Д А Ю
Должность, компания
___________________ ФИО 
«____» _____________________20__ г.
ТЕХНИЧЕСКОЕ ЗАДАНИЕ
на приобретение программного обеспечения - Системы «Электронная оперативная докум...
✅ Инициализирован анализатор VseGPT
   Модель: openai/gpt-4o-mini
   Base URL: https://api.vsegpt.ru/v1

📤 Отправляю запрос к VseGPT...
   Количество критериев: 26
   Размер 

In [18]:
#@title Запрашиваем ключ API пользователя и устанавливаем его как переменную окружения
openai_key = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = openai_key

In [20]:
#@title Демо запроса
client = OpenAI(
    api_key=openai_key, # ваш ключ в VseGPT после регистрации
    base_url="https://api.vsegpt.ru/v1",

)

messages = [
  {"role": "system", "content": "Ты - большая языковая модель. Отвечай на вопросы пользователя."},
  {"role": "user", "content": "Кто убил Кеннеди?"}]
completion = client.chat.completions.create(
    model='openai/gpt-4o-mini',
    messages=messages,
    temperature=0.1,
    extra_headers={ "X-Title": "Colab Base Example" }, # опционально - передача информация об источнике API-вызова
)
answer = completion.choices[0].message.content

print(answer)


Убийство президента США Джона Ф. Кеннеди произошло 22 ноября 1963 года в Далласе, штат Техас. Официальное расследование, проведенное Комиссией Уоррена, пришло к выводу, что убийцей был Ли Харви Освальд, который действовал в одиночку. Однако это событие породило множество теорий заговора и споров, и до сих пор существуют различные мнения о том, что на самом деле произошло.


In [23]:
!pip install -q openai==1.63.0 langchain==0.0.335 tiktoken==0.5.1 faiss-cpu==1.7.4

ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.12.0, 1.13.0, 1.13.1)
ERROR: No matching distribution found for faiss-cpu==1.7.4
